# EYES-DEFY-ANEMIA -- Phase 4 Classification (v2, CLEAN DATA) -- CNN Architectures

6 CNN architectures x 2 tissue types = 12 combos (`resnet18`, `mobilenet_v3_small`, `efficientnet_b0`, `densenet121`, `convnext_tiny`, `regnet_y_400mf`), retrained against the reprocessed dataset after the white-background bug fix (2026-08-01). Queued for Kaggle's "Save Version -> Save & Run All" background execution, cheapest architecture first.

The 3 transformer architectures (`swin_t`, `vit_b_16`, `vit_l_16`) are deliberately **not** included here -- run in a separate session instead (`classification-vit-clean.ipynb`), since `vit_b_16`/`vit_l_16` are far more compute-expensive than any CNN in this roster (86.6M / 304.3M params vs. under 30M for all 6 CNNs here). `swin_t` is grouped with the transformers, not the CNNs, despite being comparatively lightweight (~28M params) -- it's a transformer architecture (local-attention), matching this project's existing `architecture_family` categorization (`classification/outputs/v2_comparison_results/comparison_table.csv`), not a weight-class split.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [ ]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
import sys

sys.path.insert(0, "classification/datapreparepipeline")
from dataset import get_dataloaders

loaders = get_dataloaders("palpebral")
images, labels, countries = next(iter(loaders["train"]))
print("Batch shape:", images.shape)
print("Train patients:", len(loaders["train"].dataset))
print("Val patients:", len(loaders["val"].dataset))

## Training -- 12 combos, clean data, cheapest architecture first

Retrains this subset of the 18-combo v2 sweep against the reprocessed dataset (white-background bug fixed, `classification/.project_memory/02_current_status.md` "Data bug fixed" entry, 2026-08-01). Each script's `model_name` carries a `_v2_clean` suffix (not a bare `_v2` re-run) specifically so these results never silently overwrite the original v2 dirty-data results under the same filename -- both stay on disk, comparable side-by-side. Same v2 protocol otherwise (100-epoch ceiling, patience=7, dropout_rate tuned, 12-trial Optuna search) -- nothing about the search itself changed, only the input images.

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/outputs/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/cnn_clean_results.zip. Called after EVERY training cell
    below, not just at the end -- if the run gets cut short partway
    through the 12 combos (a real possibility on a long unattended Save &
    Run All), whatever completed so far is still cleanly consolidated and
    zipped, ready to download, rather than only existing nested several
    directories deep with no single downloadable archive."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/cnn_clean_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

In [ ]:
# Clean 1/12 -- RegNetY-400MF, palpebral
!python classification/v2_clean_scripts/train_regnet_y_400mf_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 2/12 -- RegNetY-400MF, forniceal_palpebral
!python classification/v2_clean_scripts/train_regnet_y_400mf_forniceal_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 3/12 -- MobileNetV3-Small, palpebral
!python classification/v2_clean_scripts/train_mobilenet_v3_small_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 4/12 -- MobileNetV3-Small, forniceal_palpebral
!python classification/v2_clean_scripts/train_mobilenet_v3_small_forniceal_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 5/12 -- EfficientNet-B0, palpebral
!python classification/v2_clean_scripts/train_efficientnet_b0_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 6/12 -- EfficientNet-B0, forniceal_palpebral
!python classification/v2_clean_scripts/train_efficientnet_b0_forniceal_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 7/12 -- ResNet18, palpebral
!python classification/v2_clean_scripts/train_resnet18_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 8/12 -- ResNet18, forniceal_palpebral
!python classification/v2_clean_scripts/train_resnet18_forniceal_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 9/12 -- DenseNet121, palpebral
!python classification/v2_clean_scripts/train_densenet121_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 10/12 -- DenseNet121, forniceal_palpebral
!python classification/v2_clean_scripts/train_densenet121_forniceal_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 11/12 -- ConvNeXt-Tiny, palpebral
!python classification/v2_clean_scripts/train_convnext_tiny_palpebral_v2_clean.py
sync_outputs()

In [ ]:
# Clean 12/12 -- ConvNeXt-Tiny, forniceal_palpebral
!python classification/v2_clean_scripts/train_convnext_tiny_forniceal_palpebral_v2_clean.py
sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever combos completed) and zipped to `/kaggle/working/cnn_clean_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip directly from there, or browse the folder for individual files.

In [ ]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/cnn_clean_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")